# Convert the waste classifier to ONNX

Run these cells in Google Colab. Upload `waste_classifier.keras` and `class_names.json` when prompted. The notebook preserves the model's internal MobileNetV2 rescaling layer and uses real uploaded image data to compare Keras and ONNX Runtime outputs before downloading the ONNX file.

The project model was saved with Keras 3.13.2. Conversion tooling compatibility can vary by TensorFlow version. If conversion fails, keep the full error output and do not use an unverified ONNX file.

In [ ]:
%pip install -q tf2onnx onnx onnxruntime

## Upload the trained model and its class order

In [ ]:
from google.colab import files

uploaded = files.upload()
required_files = {"waste_classifier.keras", "class_names.json"}
missing_files = required_files - set(uploaded)
if missing_files:
    raise FileNotFoundError(f"Upload these files: {sorted(missing_files)}")

KERAS_PATH = "/content/waste_classifier.keras"
CLASS_NAMES_PATH = "/content/class_names.json"
ONNX_PATH = "/content/waste_classifier.onnx"
SAVED_MODEL_PATH = "/content/waste_saved_model"

## Load and verify the trained Keras model

The expected input is NHWC `(224, 224, 3)`. The Keras graph must contain the internal Rescaling layer, and its output count must match the six labels.

In [ ]:
import json
import shutil
import subprocess
import sys

import numpy as np
import tensorflow as tf

EXPECTED_CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as class_file:
    class_names = json.load(class_file)
assert class_names == EXPECTED_CLASSES, f"Unexpected class order: {class_names}"

model = tf.keras.models.load_model(KERAS_PATH, compile=False)
assert tuple(model.input_shape) == (None, 224, 224, 3), model.input_shape
assert model.output_shape[-1] == len(class_names), model.output_shape
rescaling_layers = [
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.Rescaling)
]
assert any(
    np.isclose(float(layer.scale), 1.0 / 127.5)
    and np.isclose(float(layer.offset), -1.0)
    for layer in rescaling_layers
), "Expected the model's internal MobileNetV2 Rescaling layer was not found."

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("Input:", model.input_shape, "Output:", model.output_shape)
print("Classes:", class_names)

## Export a serving signature and convert it

The serving signature accepts float32 NHWC pixels in the original 0–255 range. It calls the original model directly, so the model's internal rescaling remains part of the exported graph.

In [ ]:
@tf.function(
    input_signature=[
        tf.TensorSpec([None, 224, 224, 3], tf.float32, name="images")
    ]
)
def serving_function(images):
    return {"probabilities": model(images, training=False)}

shutil.rmtree(SAVED_MODEL_PATH, ignore_errors=True)
tf.saved_model.save(
    model,
    SAVED_MODEL_PATH,
    signatures={"serving_default": serving_function},
)

conversion_command = [
    sys.executable,
    "-m",
    "tf2onnx.convert",
    "--saved-model",
    SAVED_MODEL_PATH,
    "--signature_def",
    "serving_default",
    "--opset",
    "17",
    "--output",
    ONNX_PATH,
]
subprocess.run(conversion_command, check=True)
print("Converted ONNX model:", ONNX_PATH)

## Verify ONNX structure and compare against Keras

Upload one real dataset image for each class when prompted (for example, `cardboard1.jpg`, `glass1.jpg`, and so on). This cell applies the same EXIF correction, RGB conversion, and resize as the Flask app, then compares all six Keras and ONNX outputs using unscaled float32 pixel values. It stops on a shape, dtype, class-order, or parity mismatch.

In [ ]:
from pathlib import Path

import onnx
import onnxruntime as ort
from PIL import Image, ImageOps

onnx.checker.check_model(ONNX_PATH)
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"],
)
onnx_inputs = session.get_inputs()
onnx_outputs = session.get_outputs()
assert len(onnx_inputs) == 1, f"Expected one ONNX input, got {len(onnx_inputs)}"
assert len(onnx_outputs) == 1, f"Expected one ONNX output, got {len(onnx_outputs)}"
onnx_input = onnx_inputs[0]
onnx_output = onnx_outputs[0]
assert onnx_input.type == "tensor(float)", onnx_input.type
assert len(onnx_input.shape) == 4, onnx_input.shape
for actual, expected in zip(onnx_input.shape[1:], (224, 224, 3)):
    assert not isinstance(actual, int) or actual == expected, onnx_input.shape
assert len(onnx_output.shape) == 2, onnx_output.shape
assert onnx_output.shape[-1] == len(class_names), onnx_output.shape
print("ONNX input:", onnx_input.name, onnx_input.shape, onnx_input.type)
print("ONNX output:", onnx_output.name, onnx_output.shape, onnx_output.type)

test_batches = []
for class_name in class_names:
    print(f"Upload one real {class_name} image (filename must start with {class_name!r}).")
    sample_upload = files.upload()
    if len(sample_upload) != 1:
        raise ValueError(f"Upload exactly one {class_name} test image.")
    filename, image_bytes = next(iter(sample_upload.items()))
    if not Path(filename).stem.lower().startswith(class_name):
        raise ValueError(
            f"Expected a {class_name} image filename, got {filename!r}."
        )
    sample_path = "/content/" + filename
    with open(sample_path, "wb") as image_file:
        image_file.write(image_bytes)
    with Image.open(sample_path) as source_image:
        sample_image = ImageOps.exif_transpose(source_image).convert("RGB")
        sample_image = sample_image.resize((224, 224), Image.Resampling.BILINEAR)
        image_batch = np.expand_dims(
            np.asarray(sample_image, dtype=np.float32), axis=0
        )
    assert image_batch.shape == (1, 224, 224, 3), image_batch.shape
    assert image_batch.dtype == np.float32, image_batch.dtype
    assert image_batch.min() >= 0 and image_batch.max() <= 255
    test_batches.append((filename, image_batch))

max_absolute_difference = 0.0
for filename, image_batch in test_batches:
    keras_result = np.asarray(model(image_batch, training=False))
    onnx_result = session.run(
        [onnx_output.name], {onnx_input.name: image_batch}
    )[0]
    assert keras_result.shape == (1, len(class_names)), keras_result.shape
    assert onnx_result.shape == keras_result.shape, onnx_result.shape
    assert np.all(np.isfinite(keras_result)) and np.all(np.isfinite(onnx_result))
    np.testing.assert_allclose(onnx_result, keras_result, rtol=1e-3, atol=1e-4)
    difference = float(np.max(np.abs(onnx_result - keras_result)))
    max_absolute_difference = max(max_absolute_difference, difference)
    scores = keras_result[0]
    if not (
        np.all(scores >= 0)
        and np.all(scores <= 1)
        and np.isclose(np.sum(scores), 1.0, rtol=1e-3, atol=1e-3)
    ):
        scores = np.exp(scores - np.max(scores))
        scores = scores / np.sum(scores)
    predicted_index = int(np.argmax(scores))
    print(
        f"{filename}: Keras/ONNX match; actual prediction="
        f"{class_names[predicted_index]}, "
        f"confidence={float(scores[predicted_index]) * 100:.2f}%"
    )

print("Keras/ONNX parity check: PASS for all six real images")
print("Maximum absolute difference:", max_absolute_difference)

## Download the verified ONNX file

Only run this cell after the ONNX structure check and real-image parity check pass. Copy the downloaded file into the project root beside `app.py`.

In [ ]:
files.download(ONNX_PATH)